**A Deep Learning Pipeline for Direct Detection of Antimicrobial Resistance (AMR) Genes from Raw Nanopore Electrical Signals.**

## 📂 Project Structure
```text
NanoSquiggle-AMR-CNN/
├── configs/            # Hyperparameters (YAML/JSON). No Python code.
├── data/               # Local data storage (Ignored by git).
│   ├── raw/            # .pod5 files (Primary format) & db_resistencia.fasta
│   └── processed/      # Aligned SAM files and normalized tensors.
│   └── index/          # Master CSV files mapping signal segments to labels.
├── docs/               # Technical specs and literature summaries.
├── notebooks/          # Exploratory Data Analysis (EDA) and Colab Notebooks.
├── scripts/            # Execution entry points.
├── src/                # The Core Engine (The "Library")
│   ├── data/           # Dataset and DataLoader classes (PyTorch).
│   ├── models/         # Modular architecture definitions (1D-CNN, ResNet).
│   └── utils/          # Bash setup scripts, extraction logic, and windowing.
├── tests/              # Unit tests for model shapes and data logic.
├── .gitignore          # Crucial: Excludes /data, .fast5, and /env.
├── requirements.txt    # Dependency list (torch, pysam, pandas, etc.).
└── README.md           # You are reading this!
```

## ⚙️ The "Geometry Engine": How it Works
To train the CNN, we must extract the exact chunk of the electrical signal that corresponds to the AMR gene. We achieve this using **Dorado's Move Tables (`mv`)**:
1. **Basecalling & Alignment:** Dorado processes the `.pod5` file and aligns it to our resistance database, generating a `.sam` file.
2. **The Stride & Moves:** We extract the `mv` (Move Table) and `ts` (Trim Start) tags from the SAM file.
3. **Signal Translation:** We mathematically map the biological start/end coordinates from Minimap2 back into the raw electrical samples, yielding precise slicing boundaries for our CNN.
4. **Dataset Balancing:** Unmapped reads are automatically extracted to serve as hard negatives (`Label = 0`), teaching the network to distinguish background genomic noise from AMR genes (`Label = 1`).

---

## 🚀 Quick Start (Google Colab)

### Prerequisites
* **Google Colab:** Go to *Runtime > Change runtime type* and select **T4 GPU**.

### Step 1: Environment Setup & Data Acquisition
This step creates the folder structure, downloads the Oxford Nanopore **Dorado** basecaller, and fetches the raw `.pod5` data for the specific bacterial strain.

* **In Colab:** Run the `01_download_data` bash cell in your notebook.
* **Important:** Ensure you have uploaded the `db_resistencia.fasta` file to `data/raw/` (or the root folder, depending on your setup) before proceeding.

### Step 2: Basecalling & Alignment
We run Dorado using the High Accuracy (`hac`) model. The `--emit-moves` parameter is strictly required, as it embeds the signal-to-base timing map into our output SAM file.

* **In Colab:** Run the `02_process_reads` bash cell. 
* *Note: This step requires a GPU and may take some time depending on the dataset size.*

### Step 3: Ground Truth Extraction (The Master CSV)
Once we have our `aligned_reads.sam` file, we run the PyTorch/Python data prep script. This will iterate through the alignments and generate our target labels.

```bash
# Example execution from the root directory
python src/utils/parse_move_table.py
```
**Output:** A `master_index.csv` containing:
* `read_id`: The unique identifier of the sequence.
* `pod5_file`: The source electrical file.
* `signal_start` / `signal_end`: The exact indices to slice the raw tensor.
* `label`: `1` (AMR Gene Present) or `0` (Background DNA).


## 🛠️ Built With
* [Dorado](https://github.com/nanoporetech/dorado) - Oxford Nanopore's official basecaller.
* [PyTorch](https://pytorch.org/) - Deep Learning Framework.
* [Pysam](https://pysam.readthedocs.io/) - Python interface for reading SAM/BAM alignment files.

In [1]:
%%bash
# 01_download_data.sh: Colab Environment Setup and Data Acquisition

# 1. Configuración de Rutas (Específico para Google Colab)
ROOT_DIR="/content/NanoSquiggle-AMR-CNN"
mkdir -p "$ROOT_DIR"
cd "$ROOT_DIR" || exit

echo "📁 Initializing project structure in: $(pwd)"
mkdir -p bin data/raw data/processed configs docs notebooks scripts tests

# 2. Download do Dorado into bin/
if [ ! -f "bin/dorado" ]; then
    echo "⬇️ Downloading Dorado..."
    curl -L "https://cdn.oxfordnanoportal.com/software/analysis/dorado-0.5.0-linux-x64.tar.gz" -o dorado.tar.gz
    echo "📦 Organizando Dorado en bin/..."
    tar -xzf dorado.tar.gz

    # Movemos los binarios y librerías
    mv dorado-0.5.0-linux-x64/bin/dorado bin/
    mv dorado-0.5.0-linux-x64/lib bin/

    # Clean files.
    rm -rf dorado-0.5.0-linux-x64 dorado.tar.gz
    echo "✅ Dorado installed in bin/"
fi

# 3. Download da Estirpe KP1055
ESTIRPE="KP1779" # Cambia a KP1779 si lo necesitas
DATA_URL="https://data.narodni-repozitar.cz/general/datasets/dj8ys-a4r49/files/${ESTIRPE}_pod5.tar.gz"
RAW_DATA_DIR="data/raw/${ESTIRPE}"

if [ ! -d "$RAW_DATA_DIR" ]; then
    echo "⬇️ Descarregando ${ESTIRPE} ..."
    mkdir -p "$RAW_DATA_DIR"
    wget -q --show-progress -O data/raw/${ESTIRPE}.tar.gz "$DATA_URL"

    # Extração e Limpeza imediata
    echo "📦 Extraindo e organizando POD5..."
    tar -xzf data/raw/${ESTIRPE}.tar.gz -C "$RAW_DATA_DIR"
    rm data/raw/${ESTIRPE}.tar.gz

    # Movemos los archivos de la carpeta extraída a la raíz de RAW_DATA_DIR
    if [ -d "$RAW_DATA_DIR/pod5_pass" ]; then
        mv "$RAW_DATA_DIR/pod5_pass"/* "$RAW_DATA_DIR/"
        rm -rf "$RAW_DATA_DIR/pod5_fail" "$RAW_DATA_DIR/pod5_pass"
    fi
    echo "✅ Dataset prepared in $RAW_DATA_DIR"
fi

# 4. Move Resistance Database
# NOTA PARA COLAB: Tus amigos deben subir 'db_resistencia.fasta' manualmente a Colab
# o debes descargarlo de un GitHub si lo tienes público.
if [ -f "files_tratment/db_resistencia.fasta" ]; then
    mv "files_tratment/db_resistencia.fasta" "data/raw/db_resistencia.fasta"
    echo "✅ Move db_resistencia.fasta to data/raw/"
else
    echo "⚠️ ATENCIÓN: db_resistencia.fasta no encontrado."
    echo "Por favor, sube el archivo a /content/NanoSquiggle-AMR-CNN/files_tratment/"
fi

echo "✅ Setup concluído com sucesso."

📁 Initializing project structure in: /content/NanoSquiggle-AMR-CNN
⬇️ Downloading Dorado...
📦 Organizando Dorado en bin/...
✅ Dorado installed in bin/
⬇️ Descarregando KP1779 ...
📦 Extraindo e organizando POD5...
✅ Dataset prepared in data/raw/KP1779
⚠️ ATENCIÓN: db_resistencia.fasta no encontrado.
Por favor, sube el archivo a /content/NanoSquiggle-AMR-CNN/files_tratment/
✅ Setup concluído com sucesso.


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2050M  100 2050M    0     0  20.9M      0  0:01:37  0:01:37 --:--:-- 15.6M

     0K .......... .......... .......... .......... ..........  0%  163K 3h0m
    50K .......... .......... .......... .......... ..........  0%  327K 2h15m
   100K .......... .......... .......... .......... ..........  0%  120M 90m14s
   150K .......... .......... .......... .......... ..........  0%  309M 67m42s
   200K .......... .......... .......... .......... ..........  0%  326K 72m12s
   250K .......... .......... .......... .......... ..........  0%  127M 60m12s
   300K .......... .......... .......... .......... ..........  0%  305M 51m37s
   350K .......... .......... .......... .......... ..........  0%  269M 45m11s
   400K .......... .......... .......... .......... ..........  0%  329K 50m7s
   450K .......... .......... .......... .....

In [2]:
%%bash
# 02_process_reads.sh: Unified Basecalling & Alignment (Colab Version)

# Abort on error
set -e

# 1. Path Configuration (Specific for Google Colab)
ROOT_DIR="/content/NanoSquiggle-AMR-CNN"
cd "$ROOT_DIR" || { echo "❌ Error: Root directory not found. Run script 01 first."; exit 1; }

# Variable Definitions
DORADO_BIN="bin/dorado"
DORADO_LIB="bin/lib"
REFERENCE="data/raw/db_resistencia.fasta"

# ---------------------------------------------------------
# ⚠️ CHANGE THIS VARIABLE TO PROCESS DIFFERENT STRAINS
ESTIRPE="KP1779" 
# ---------------------------------------------------------

POD5_DIR="data/raw/${ESTIRPE}"
OUTPUT_SAM="data/processed/${ESTIRPE}/aligned_reads.sam"

# Create necessary directories
mkdir -p data/processed/${ESTIRPE}

# 2. File Validation
if [ ! -f "$DORADO_BIN" ]; then
    echo "❌ Error: Dorado binary not found in bin/. Did you run the download cell?"
    exit 1
fi

if [ ! -f "$REFERENCE" ]; then
    echo "❌ Error: FASTA file not found at $REFERENCE."
    echo "Make sure to upload the db_resistencia.fasta file to Colab."
    exit 1
fi

if [ ! -d "$POD5_DIR" ]; then
    echo "❌ Error: POD5 data folder not found at $POD5_DIR."
    exit 1
fi

# 3. CRUCIAL CONFIGURATION: Load shared libraries
export LD_LIBRARY_PATH="$ROOT_DIR/bin/lib:$LD_LIBRARY_PATH"

echo "🧬 Starting Unified Basecalling & Alignment (Dorado)..."
echo "🎯 Reference: $REFERENCE"
echo "📁 Data Source: $POD5_DIR"
echo "💾 Output File: $OUTPUT_SAM"

# 4. Dorado Execution
# --emit-moves: Fundamental for mapping signal to bases (Geometry Engine)
# --emit-sam: Standard alignment format
# 'hac' model (High Accuracy). It will be downloaded automatically the first time.

$DORADO_BIN basecaller hac "$POD5_DIR" \
    --reference "$REFERENCE" \
    --emit-moves \
    --emit-sam > "$OUTPUT_SAM"

if [ $? -eq 0 ]; then
    echo "✅ Pipeline complete. Results saved to $OUTPUT_SAM"
else
    echo "❌ Error during Dorado execution."
    exit 1
fi

❌ Error: FASTA file not found at data/raw/db_resistencia.fasta.
Make sure to upload the db_resistencia.fasta file to Colab.


CalledProcessError: Command 'b'# 02_process_reads.sh: Unified Basecalling & Alignment (Colab Version)\n\n# Abort on error\nset -e\n\n# 1. Path Configuration (Specific for Google Colab)\nROOT_DIR="/content/NanoSquiggle-AMR-CNN"\ncd "$ROOT_DIR" || { echo "\xe2\x9d\x8c Error: Root directory not found. Run script 01 first."; exit 1; }\n\n# Variable Definitions\nDORADO_BIN="bin/dorado"\nDORADO_LIB="bin/lib"\nREFERENCE="data/raw/db_resistencia.fasta"\n\n# ---------------------------------------------------------\n# \xe2\x9a\xa0\xef\xb8\x8f CHANGE THIS VARIABLE TO PROCESS DIFFERENT STRAINS\nESTIRPE="KP1779" \n# ---------------------------------------------------------\n\nPOD5_DIR="data/raw/${ESTIRPE}"\nOUTPUT_SAM="data/processed/${ESTIRPE}/aligned_reads.sam"\n\n# Create necessary directories\nmkdir -p data/processed/${ESTIRPE}\n\n# 2. File Validation\nif [ ! -f "$DORADO_BIN" ]; then\n    echo "\xe2\x9d\x8c Error: Dorado binary not found in bin/. Did you run the download cell?"\n    exit 1\nfi

In [ ]:
import pysam
import pandas as pd
import random
from pathlib import Path

def generate_production_dataset(bam_path, output_csv, neg_ratio=3, coverage_threshold=95.0):
    print(f"🧬 Opening Sorted BAM file: {bam_path}")
    
    # "rb" stands for Read Binary (Required for BAM files)
    samfile = pysam.AlignmentFile(bam_path, "rb") 
    
    pos_records = []
    neg_records = []
    discarded_weak = 0

    print("🧐 Scanning for High-Quality Positives and Hard Negatives...")

    for read in samfile.fetch(until_eof=True):
        tags = dict(read.tags)
        
        # 1. QUALITY CHECKER: Must have Move Table and Trim Start
        if 'mv' not in tags or 'ts' not in tags:
            continue
            
        stride = tags['mv'][0]      # The dynamic stride (e.g., 6)
        moves = tags['mv'][1:]      # The 1s and 0s array
        trim_start = tags['ts']     # Skipped electrical samples
        pod5_filename = tags.get('fn', 'unknown.pod5')

        # --- CLASS 1: High-Quality Mapped Reads (The Positives) ---
        if not read.is_unmapped and not read.is_secondary:
            
            # 2. FILTER FOR WEAKS: Calculate actual coverage percentage
            gene_length = samfile.get_reference_length(read.reference_name)
            alignment_length = read.reference_length # How many bases actually matched
            
            coverage_pct = (alignment_length / gene_length) * 100
            
            if coverage_pct < coverage_threshold:
                discarded_weak += 1
                continue # Skip this read, it's too fragmented
            
            # 3. SPATIAL GEOMETRY: Calculate exact signal coordinates
            base_start = read.query_alignment_start
            base_end = read.query_alignment_end
            
            moves_to_start = 0
            base_count = 0
            for step in moves:
                base_count += step
                moves_to_start += 1
                if base_count == base_start:
                    break
                    
            moves_to_end = moves_to_start
            for step in moves[moves_to_start:]:
                base_count += step
                moves_to_end += 1
                if base_count == base_end:
                    break

            signal_start = trim_start + (moves_to_start * stride)
            signal_end = trim_start + (moves_to_end * stride)

            pos_records.append({
                "read_id": read.query_name,
                "pod5_file": pod5_filename,
                "target_gene": read.reference_name,
                "signal_start": signal_start, 
                "signal_end": signal_end,
                "label": 1,
                "coverage_pct": round(coverage_pct, 2)
            })

        # --- CLASS 0: Unmapped Reads (The Hard Negatives) ---
        elif read.is_unmapped:
            # Standard 30,000 sample window of pure Klebsiella background
            neg_records.append({
                "read_id": read.query_name,
                "pod5_file": pod5_filename,
                "target_gene": "NONE_BACKGROUND",
                "signal_start": 10000, 
                "signal_end": 40000,
                "label": 0,
                "coverage_pct": 0.0
            })

    samfile.close()

    # --- DATASET BALANCING ---
    print(f"\n📊 Extracted {len(pos_records)} High-Quality Positives (>95% Coverage).")
    print(f"🗑️  Discarded {discarded_weak} weak/fragmented alignments.")
    
    # We enforce a strict Negative:Positive ratio so the model doesn't get biased
    num_neg_to_keep = len(pos_records) * neg_ratio
    
    if len(neg_records) > num_neg_to_keep:
        neg_records = random.sample(neg_records, num_neg_to_keep)
        
    print(f"⚖️  Kept {len(neg_records)} Hard Negatives to maintain a {neg_ratio}:1 ratio.")

    # Combine, Shuffle randomly, and Save
    final_df = pd.DataFrame(pos_records + neg_records)
    
    # If the dataframe is not empty, shuffle and save
    if not final_df.empty:
        final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
        final_df.to_csv(output_csv, index=False)
        print(f"\n✅ SUCCESS: Master CSV written to '{output_csv}' with {len(final_df)} total rows.")
    else:
        print("\n❌ ERROR: No valid data found to write to CSV.")


In [ ]:
if __name__ == "__main__":
    # --- CONFIGURACIÓN ESPECÍFICA PARA GOOGLE COLAB ---
    # En Colab, __file__ no está definido, así que fijamos la raíz manualmente
    from pathlib import Path
    
    ROOT_DIR = Path("/content/NanoSquiggle-AMR-CNN")
    
    # Verificación de seguridad
    if not ROOT_DIR.exists():
        print(f"❌ Error: No se encontró la ruta {ROOT_DIR}. ¿Ejecutaste las celdas de Bash primero?")
    else:
        # Definir la estirpe
        ESTIRPE = "KP1779" # Cambia a KP1055 si lo necesitas
        
        # Construir las rutas exactas
        BAM_FILE = ROOT_DIR / "data" / "processed" / ESTIRPE / f"aligned_sorted_{ESTIRPE}.bam"
        OUT_CSV = ROOT_DIR / "data" / "master.csv"
        
        # Ejecutar la extracción
        generate_production_dataset(str(BAM_FILE), str(OUT_CSV))